# Fine Tuning eficiente com LoRA — EscutIA

Nesta etapa vamos entender e executar um primeiro Supervised Fine Tuning usando LoRA e LLaMA-Factory. O objetivo é adaptar um modelo pequeno para classificar textos em português como negativo, neutro ou positivo.

O notebook é didático e deve ser executado célula por célula. Ele não executa treinamento automaticamente: a célula de treinamento pede uma confirmação explícita.

## O que será aprendido

Ao final desta etapa, você deverá conseguir:

- explicar adapters, PEFT e LoRA;
- identificar o que fica congelado e o que é treinado;
- configurar rank, alpha, dropout e módulos alvo;
- executar SFT com LoRA no LLaMA-Factory;
- acompanhar loss, checkpoints e sinais de sobreajuste;
- analisar como os hiperparâmetros influenciam o treinamento;
- carregar o adapter produzido para uso posterior.

## Pré-requisitos

Antes de continuar, execute o Dataset_EscutIA.ipynb e o Analise_DataSet_EscutIA.ipynb. O treinamento só deve ser considerado quando o relatório final indicar DATA_READY_FOR_SFT.

Nesta etapa ainda não usamos quantização em 4 bits. QLoRA será estudado em um módulo separado.

In [1]:
from pathlib import Path
import json
import shutil
import matplotlib.pyplot as plt
from IPython.display import display

BASE_DIR = Path.cwd()
if not (BASE_DIR / 'configs').exists() and (BASE_DIR / 'EscutIA' / 'fine_tuning_lora' / 'configs').exists():
    BASE_DIR = BASE_DIR / 'EscutIA' / 'fine_tuning_lora'

DATASET_DIR = (BASE_DIR.parent / 'dataset' / 'dados' / 'preparados').resolve()
CONFIG_PATH = BASE_DIR / 'configs' / 'lora_escutia.yaml'
GATE_PATH = DATASET_DIR.parent / 'relatorios' / '11_validacao_final.json'
OUTPUT_DIR = BASE_DIR / 'outputs' / 'resultados' / 'lora_escutia'
LOG_DIR = BASE_DIR / 'outputs' / 'logs' / 'lora_escutia'

arquivos_dataset = [
    DATASET_DIR / 'dataset_info.json',
    DATASET_DIR / 'escutia_train.json',
    DATASET_DIR / 'escutia_validation.json',
]
ausentes = [str(caminho) for caminho in arquivos_dataset if not caminho.exists()]
if ausentes:
    raise FileNotFoundError('Dataset preparado incompleto. Execute primeiro o notebook de preparação: ' + '; '.join(ausentes))

gate = json.loads(GATE_PATH.read_text(encoding='utf-8')) if GATE_PATH.exists() else {}
print(f'Dataset preparado: {DATASET_DIR}')
print(f'Gate atual: {gate.get("decisao", "não encontrado")}')
print(f'Configuração LoRA: {CONFIG_PATH}')
print(f'LLaMA-Factory disponível: {shutil.which("llamafactory-cli") is not None}')


Dataset preparado: C:\Users\mdbaa\development\alura\alura-llama-factory\EscutIA\dataset\dados\preparados
Gate atual: DATA_READY_FOR_SFT
Configuração LoRA: c:\Users\mdbaa\development\alura\alura-llama-factory\EscutIA\fine_tuning_lora\configs\lora_escutia.yaml
LLaMA-Factory disponível: True


## 1. O que são PEFT e adapters?

No Fine Tuning completo, os pesos do modelo inteiro podem ser atualizados. Isso exige muita memória para armazenar gradientes, estados do otimizador e checkpoints.

PEFT significa Parameter-Efficient Fine Tuning. A ideia é manter o modelo base congelado e treinar apenas uma pequena quantidade de parâmetros adicionais. Esses parâmetros adicionais são o adapter.

O adapter pode ser salvo separadamente e combinado com o modelo base durante a inferência. Assim, não precisamos duplicar todos os pesos do modelo para cada especialização.

## 2. Princípio da Low Rank Adaptation

O LoRA não substitui diretamente a matriz original do modelo. Ele aprende uma atualização de baixa dimensão para alguns módulos escolhidos:

W_adaptado = W_base + escala × B × A

O peso W_base permanece congelado. As matrizes A e B são pequenas e treináveis. O rank define a capacidade dessas matrizes: rank maior oferece mais capacidade, mas aumenta parâmetros, memória e risco de sobreajuste.

Nesta primeira configuração vamos adaptar os módulos de atenção q_proj, k_proj, v_proj e o_proj.

In [2]:
parametros_lora = {
    'rank': 8,
    'alpha': 16,
    'dropout': 0.05,
    'módulos alvo': ['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    'modelo base': 'Qwen/Qwen2.5-0.5B-Instruct',
    'estratégia': 'SFT + LoRA',
}
display(__import__('pandas').DataFrame([parametros_lora]))


,rank,alpha,dropout,módulos alvo,modelo base,estratégia
0,8,16,0.05,"[q_proj, k_proj, v_proj, o_proj]",Qwen/Qwen2.5-0.5B-Instruct,SFT + LoRA


## 3. O que será treinado?

Durante o LoRA, os pesos originais do modelo base ficam congelados. Os parâmetros treináveis são os adapters associados aos módulos alvo.

A célula seguinte é uma inspeção opcional. Ela carrega o modelo e aplica uma configuração LoRA para mostrar a quantidade de parâmetros treináveis. Execute-a somente depois de preparar o ambiente do LLaMA-Factory e autorizar o carregamento do modelo.

In [3]:
from transformers import AutoModelForCausalLM
from peft import LoraConfig, TaskType, get_peft_model

MODEL_NAME = 'Qwen/Qwen2.5-0.5B-Instruct'
MODEL_REVISION = '7ae557604adf67be50417f59c2c2f167def9a775'

modelo_base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    revision=MODEL_REVISION,
    torch_dtype='auto',
)
config_lora = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
)
modelo_com_lora = get_peft_model(modelo_base, config_lora)
modelo_com_lora.print_trainable_parameters()


`torch_dtype` is deprecated! Use `dtype` instead!


trainable params: 1,081,344 || all params: 495,114,112 || trainable%: 0.2184


## 4. Hiperparâmetros da primeira configuração

A configuração inicial é conservadora para um primeiro experimento:

- microbatch de 1 para controlar o pico de memória;
- acumulação de gradiente igual a 8, produzindo batch efetivo 8;
- learning rate de 1e-4, comum para adapters LoRA;
- duas épocas para observar o comportamento sem prolongar desnecessariamente o treinamento;
- cutoff de 256 tokens para manter o custo previsível;
- avaliação e checkpoint ao final de cada época.

Esses valores são uma configuração inicial, não uma verdade universal. O hardware e o comportamento da loss devem orientar os ajustes.

In [4]:
config_texto = CONFIG_PATH.read_text(encoding='utf-8')
print(config_texto)

batch_por_dispositivo = 1
acumulacao = 8
print(f'Batch efetivo planejado: {batch_por_dispositivo * acumulacao}')


# Configuração didática do primeiro experimento de SFT com LoRA.
# O treinamento só deve ser executado depois que o dataset tiver DATA_READY_FOR_SFT
# e o ambiente/modelo tiverem sido validados.

model_name_or_path: Qwen/Qwen2.5-0.5B-Instruct
model_revision: 7ae557604adf67be50417f59c2c2f167def9a775

dataset_dir: ../dataset/dados/preparados
dataset: escutia_treino
eval_dataset: escutia_validacao

stage: sft
finetuning_type: lora
template: qwen
do_train: true
do_eval: true
seed: 42

cutoff_len: 256
per_device_train_batch_size: 1
per_device_eval_batch_size: 1
gradient_accumulation_steps: 8
learning_rate: 0.0001
num_train_epochs: 2.0
lr_scheduler_type: cosine
warmup_ratio: 0.05
optim: adamw_torch
weight_decay: 0.01
max_grad_norm: 1.0

lora_target: q_proj,k_proj,v_proj,o_proj
lora_rank: 8
lora_alpha: 16
lora_dropout: 0.05

bf16: true
fp16: false
pure_bf16: false
gradient_checkpointing: true
dataloader_num_workers: 0

eval_strategy: epoch
save_strategy: epoch
save_total_limit: 2
save_only_mo

## 5. Validar o gate antes do treinamento

O treinamento não deve começar se o dataset não estiver pronto. Esta célula faz uma verificação explícita para evitar iniciar o LoRA sobre dados ainda não validados.

In [5]:
if gate.get('decisao') != 'DATA_READY_FOR_SFT':
    raise RuntimeError('Treinamento bloqueado: o dataset ainda não está em DATA_READY_FOR_SFT.')
if not CONFIG_PATH.exists():
    raise FileNotFoundError(f'Configuração não encontrada: {CONFIG_PATH}')
print('Gate aprovado. A configuração e o dataset estão disponíveis para revisão.')


Gate aprovado. A configuração e o dataset estão disponíveis para revisão.


## 6. Executar o SFT com LoRA no LLaMA-Factory

A célula chama diretamente o comando canônico do LLaMA-Factory. Cada execução inicia o treinamento e consome tempo, memória e pode gerar checkpoints.

Se ocorrer OOM, loss não finita, fallback para CPU ou conflito no diretório de saída, pare o experimento e registre o motivo. Não reduza parâmetros silenciosamente.

In [6]:
import os
import subprocess

llama_cli = shutil.which('llamafactory-cli')
if llama_cli is None:
    raise FileNotFoundError('llamafactory-cli não foi encontrado no ambiente selecionado pelo notebook.')

ambiente = os.environ.copy()
caminho_python = str(BASE_DIR / 'scripts')
ambiente['PYTHONPATH'] = caminho_python + os.pathsep + ambiente.get('PYTHONPATH', '')
resultado = subprocess.run(
    [llama_cli, 'train', str(CONFIG_PATH)],
    cwd=BASE_DIR,
    check=False,
    capture_output=True,
    text=True,
    env=ambiente,
)
if resultado.stdout:
    print(resultado.stdout, end='')
if resultado.stderr:
    print(resultado.stderr, end='')
print(f'Treinamento finalizado com código: {resultado.returncode}')


Treinamento finalizado com código: 1


## 7. Acompanhar loss e checkpoints

Depois do treinamento, os logs ajudam a observar se a loss caiu, se a validação acompanhou o treino e se há sinais de sobreajuste. Os checkpoints permitem recuperar estados intermediários sem perder todo o trabalho.

In [7]:
import pandas as pd

log_path = OUTPUT_DIR / 'trainer_log.jsonl'
if log_path.exists():
    registros_log = [json.loads(linha) for linha in log_path.read_text(encoding='utf-8').splitlines() if linha.strip()]
    df_log = pd.DataFrame(registros_log)
    display(df_log.tail())

    colunas_loss = [coluna for coluna in ['loss', 'eval_loss'] if coluna in df_log.columns]
    if colunas_loss:
        ax = df_log.plot(x='step', y=colunas_loss, figsize=(10, 5), marker='o')
        ax.set_title('Evolução da loss')
        ax.set_ylabel('Loss')
        ax.set_xlabel('Step')
        plt.tight_layout()
        plt.show()
else:
    print(f'Log ainda não encontrado: {log_path}')

checkpoints = sorted(OUTPUT_DIR.glob('checkpoint-*')) if OUTPUT_DIR.exists() else []
print('Checkpoints encontrados:')
for checkpoint in checkpoints:
    print('-', checkpoint)


Log ainda não encontrado: c:\Users\mdbaa\development\alura\alura-llama-factory\EscutIA\fine_tuning_lora\outputs\resultados\lora_escutia\trainer_log.jsonl
Checkpoints encontrados:


## 8. Como analisar o impacto dos hiperparâmetros

Para comparar configurações de forma confiável, altere uma variável por vez e mantenha o restante fixo. Registre a configuração, a loss, o tempo, o consumo de memória e o resultado na avaliação.

| Hiperparâmetro | Quando aumentar | Possível custo |
|---|---|---|
| rank | quando o adapter não tem capacidade suficiente | mais parâmetros e risco de sobreajuste |
| alpha | quando a atualização LoRA precisa de maior escala | mudanças mais fortes no comportamento |
| dropout | quando há sinais de sobreajuste | treinamento mais regularizado, mas possivelmente mais lento |
| learning rate | quando a adaptação está muito lenta | instabilidade ou perda de conhecimento |
| epochs | quando ainda há espaço para aprender | sobreajuste e maior tempo |
| batch efetivo | quando o gradiente está muito instável | maior memória ou mais acumulação |

Uma única execução não prova qual configuração é melhor. Ela fornece uma primeira evidência para o ciclo controlado de treinamento e avaliação.

## Resultado desta etapa

Ao concluir o treinamento, teremos um adapter LoRA em outputs/resultados/lora_escutia e os checkpoints correspondentes. O adapter depende do modelo base para ser carregado.

O próximo notebook, de inferência, poderá carregar modelo base + adapter e testar novos textos. Essa demonstração não substitui a avaliação formal com o conjunto de avaliação congelado.